# Outliers de DBSCAN: serial, P1 y P2
Compila las tres versiones, las corre sobre `10000_data.csv` y grafica los outliers (etiqueta 0) de cada una.

In [ ]:
import os, subprocess
import numpy as np
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) != "dbscan":
    os.chdir("dbscan")

N = 10000
THREADS = os.cpu_count()
K = 4

data = f"data/{N}_data.csv"
if not os.path.exists(data):
    subprocess.run(["python3", "generate_data.py", str(N)], check=True)
subprocess.run(["make", "-B"], check=True)

runs = {
    "Serial": ["./dbscan_serial", data, f"data/{N}_results_serial.csv"],
    "P1":     ["./dbscan_p1",     data, f"data/{N}_results_p1.csv", str(THREADS)],
    "P2":     ["./dbscan_p2",     data, f"data/{N}_results_p2.csv", str(THREADS), str(K)],
}
for name, cmd in runs.items():
    seconds = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    print(f"{name}: {seconds} s")

In [ ]:
results = {name: np.loadtxt(cmd[2], delimiter=",") for name, cmd in runs.items()}

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (name, r) in zip(axes, results.items()):
    noise = r[:, 2] == 0
    ax.scatter(r[~noise, 0], r[~noise, 1], c="lightgray", s=8, label="clusters")
    ax.scatter(r[noise, 0], r[noise, 1], c="red", s=25, label="outliers")
    ax.set_title(f"{name}: {noise.sum()} outliers")
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="upper right")
plt.suptitle(f"Outliers detectados con DBSCAN (n={N})")
plt.tight_layout()
plt.show()

In [ ]:
masks = [r[:, 2] == 0 for r in results.values()]
print("Las 3 versiones coinciden:", all((masks[0] == m).all() for m in masks[1:]))